# GBM Examples Notebook

Original QMCPy demo: [`QMCPy/demos/GBM/gbm_examples.ipynb`](../../../QMCPy/demos/GBM/gbm_examples.ipynb)

The Python notebook compares helper modules and MAE plots for GBM samplers. This Julia translation keeps the same theme by comparing terminal-value errors and an Asian-option estimate with built-in `QMC.jl` tools.


In [1]:
using QMC
using Statistics
using Printf


## Configuration

We work with a geometric Brownian motion model

$$S(t) = S_0 \exp\!\left((\gamma - 	\frac{\sigma^2}{2}) t + \sigma W(t)
\right),$$

and compare how the sampling error changes with the number of simulated paths.


In [2]:
S0 = 100.0
γ = 0.05
σ2 = 0.04
T = 1.0
d = 8
ns = 2 .^ (7:11)
exact_terminal_mean = S0 * exp(γ * T)

function terminal_mean_error(dd)
    tm = GeometricBrownianMotion(dd; t_final=T, initial_value=S0, drift=γ, diffusion=σ2)
    x = transform(tm, gen_samples(dd, ns[end]))
    return [abs(mean(x[1:n, end]) - exact_terminal_mean) for n in ns]
end

iid_errors = terminal_mean_error(IIDStdUniform(d; seed=7))
lattice_errors = terminal_mean_error(Lattice(d; seed=7))
net_errors = terminal_mean_error(DigitalNetB2(d; seed=7, graycode=false))

println("path counts = ", ns)
println("IID errors = ", round.(iid_errors, digits=4))
println("Lattice errors = ", round.(lattice_errors, digits=4))
println("Digital net errors = ", round.(net_errors, digits=4))


path counts = [128, 256, 512, 1024, 2048]
IID errors = [2.2195, 1.4537, 0.1364, 0.0399, 0.3339]
Lattice errors = [0.089, 0.1826, 0.1335, 0.0092, 0.0297]
Digital net errors = [0.0564, 0.3294, 0.1147, 0.0139, 0.0417]


## MAE vs Number of Paths

The exact mean of the terminal stock price is available in closed form, so the absolute error of the empirical terminal mean plays the role of the MAE comparison used in the Python notebook.


In [3]:
@assert minimum(lattice_errors) < maximum(iid_errors)
@assert minimum(net_errors) < maximum(iid_errors)


## Asian Option Example

As a second GBM-based example, we price a geometric-average Asian call option using a randomized lattice rule.


In [4]:
tm_option = BrownianMotion(Lattice(12; seed=7); t_final=1.0)
asian_call = FinancialOption(tm_option; option_type=:asian, call_put=:call, mean_type=:geometric, volatility=0.5, start_price=30.0, strike_price=25.0, interest_rate=0.0)
result_option = integrate(CubQMCLatticeG(asian_call; abs_tol=5e-3))
@printf("Geometric Asian call price estimate = %.6f using n_total = %d
", result_option.solution, result_option.data[:n_total])
@printf("Closed-form benchmark used by QMC.jl = %.6f
", get_exact_value(asian_call))

@assert abs(result_option.solution - get_exact_value(asian_call)) < 0.1


Geometric Asian call price estimate = 5.953171 using n_total = 65536
Closed-form benchmark used by QMC.jl = 5.951824
